# Melanoma Detection — Transfer Learning with MobileNetV2

This notebook implements the pipeline:

1. Load and prepare data (resize, normalize, split, augment)
2. Load MobileNetV2 base (pretrained on ImageNet, frozen)
3. Add a classification head (pooling, dropout, dense output)
4. Train the model (compile, fit, monitor validation loss)
5. Evaluate the model (confusion matrix, ROC curve, AUC)
6. A minimal image-upload front end for testing single images

**Assumption about your data:** your Google Drive folder `AIMI Dataset Images` is expected to contain
one subfolder per class. For a 3-class setup that looks like:

```
AIMI Dataset Images/
    class_a/
        img001.jpg
        ...
    class_b/
        img002.jpg
        ...
    class_c/
        img003.jpg
        ...
```

Keras reads the class names from the folder names (sorted alphabetically) and the code below adapts to
however many classes it finds — nothing has the number 3 hardcoded, so this works for 2, 3, or more.


## 0. Mount Google Drive & imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, roc_auc_score, classification_report,
)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


## 1. Load and prepare data
Resize, normalize, split, augment

- **Resize**: all images are resized to `IMG_SIZE` on load.
- **Normalize**: pixel scaling is done later, inside the model, via `mobilenet_v2.preprocess_input`
  (this keeps the exact preprocessing MobileNetV2 was trained with).
- **Split**: 70% train / 15% validation / 15% test, done by first splitting off 30% as a held-out set,
  then splitting that in half into validation and test.
- **Augment**: random flip/rotation/zoom/contrast, applied only to the training set.

`label_mode='int'` gives integer labels (0, 1, 2, ...), which pairs with
`sparse_categorical_crossentropy` during training.


In [ ]:
DATA_DIR = '/content/drive/MyDrive/AIMI Dataset Images'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 123

# 70% train
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.3,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
)

# 30% held out -> split into validation + test below
holdout_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.3,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print("Classes:", class_names)
print("Number of classes:", NUM_CLASSES)

holdout_batches = holdout_ds.cardinality().numpy()
val_ds = holdout_ds.take(holdout_batches // 2)
test_ds = holdout_ds.skip(holdout_batches // 2)

print(f"Train batches: {train_ds.cardinality().numpy()}, "
      f"Val batches: {val_ds.cardinality().numpy()}, "
      f"Test batches: {test_ds.cardinality().numpy()}")


In [ ]:
# Check how many images are in each class - useful for spotting imbalance
import pathlib

data_path = pathlib.Path(DATA_DIR)
for name in class_names:
    n = len(list((data_path / name).glob('*')))
    print(f"{name}: {n} images")


In [ ]:
# Preview a few images
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(min(9, images.shape[0])):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")


In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="data_augmentation")

AUTOTUNE = tf.data.AUTOTUNE

# Augmentation is applied only to the training set. Validation/test stay untouched
# so evaluation reflects real, unaugmented images.
train_ds = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=AUTOTUNE,
).prefetch(AUTOTUNE)

val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)


## 2. Load MobileNetV2 base
Pretrained on ImageNet, frozen


In [ ]:
IMG_SHAPE = IMG_SIZE + (3,)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SHAPE,
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False  # freeze the pretrained base

base_model.summary()


## 3. Add classification head
Pooling, dropout, dense output

For multi-class the output layer has one unit per class with a **softmax** activation, so the outputs
form a probability distribution that sums to 1 across all classes.


In [ ]:
inputs = tf.keras.Input(shape=IMG_SHAPE)

# MobileNetV2's own preprocessing (scales pixels to [-1, 1])
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.summary()


## 4. Train the model
Compile, fit, monitor validation loss

`sparse_categorical_crossentropy` is the multi-class counterpart to binary crossentropy, and it expects
the integer labels produced by `label_mode='int'`.

If your classes are imbalanced (check the counts printed in section 1), uncomment the `class_weight`
block so the rarer classes aren't ignored by the model.


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        '/content/drive/MyDrive/melanoma_mobilenetv2_best.keras',
        monitor='val_loss', save_best_only=True,
    ),
]

# Optional: weight the loss by class frequency if your dataset is imbalanced.
# from sklearn.utils.class_weight import compute_class_weight
# train_labels = np.concatenate([y.numpy() for _, y in train_ds])
# weights = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=train_labels)
# class_weight = dict(enumerate(weights))
# print("Class weights:", class_weight)

EPOCHS = 20

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    # class_weight=class_weight,  # uncomment together with the block above
)


In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(len(loss))

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Loss')

plt.show()


## 5. Evaluate the model
Confusion matrix, ROC curve, AUC

Evaluation runs on `test_ds` — the held-out split the model never saw during training or
validation-based early stopping.

With more than two classes, a single ROC curve no longer applies. The standard approach is
**one-vs-rest**: for each class, treat it as the positive class and everything else as negative,
giving one curve and one AUC per class, plus a macro-average across them.


In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}")


In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)
y_pred_prob = model.predict(test_ds)          # shape: (n_samples, NUM_CLASSES)
y_pred = np.argmax(y_pred_prob, axis=1)       # highest-probability class

print(classification_report(y_true, y_pred, target_names=class_names))


In [ ]:
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(cmap='Blues', ax=ax, xticks_rotation=45)
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()


In [ ]:
# One-vs-rest ROC curve: one curve per class
y_true_bin = label_binarize(y_true, classes=np.arange(NUM_CLASSES))

plt.figure(figsize=(7, 6))
for i, name in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_prob[:, i])
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc(fpr, tpr):.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves (One-vs-Rest)')
plt.legend(loc='lower right')
plt.show()

macro_auc = roc_auc_score(y_true, y_pred_prob, multi_class='ovr', average='macro')
print(f"Macro-average one-vs-rest AUC: {macro_auc:.4f}")


## 6. Try it: minimal image-upload front end

A small web UI that renders **inline in this notebook's output** — drag in a lesion image (or click to
browse) and get back the predicted class with a confidence score for every class. It calls the same
`model` and `class_names` built above, so run this after training (or after loading a saved model).

**Disclaimer:** this is a demo interface for a student/research project, not a medical device. It must
not be used to make real diagnostic decisions.


In [ ]:
!pip install -q gradio


In [ ]:
# If you're running this section in a fresh runtime (no training done this session),
# load the model you saved during training instead of retraining:
# model = tf.keras.models.load_model('/content/drive/MyDrive/melanoma_mobilenetv2_best.keras')


In [ ]:
import gradio as gr
from PIL import Image


def predict_image(img):
    if img is None:
        return None

    image = Image.fromarray(img).convert('RGB').resize(IMG_SIZE)
    arr = np.expand_dims(np.array(image, dtype=np.float32), axis=0)  # (1, H, W, 3), pixels 0-255

    probs = model.predict(arr, verbose=0)[0]  # model already includes its own preprocessing
    return {name: float(probs[i]) for i, name in enumerate(class_names)}


demo = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type='numpy', label='Upload a skin lesion image'),
    outputs=gr.Label(num_top_classes=NUM_CLASSES, label='Prediction'),
    title='Melanoma Detection (Demo)',
    description=(
        'Upload a skin lesion image to get a classification and confidence score. '
        'Educational/demo use only — not a medical diagnostic tool.'
    ),
)

demo.launch()  # renders inline in the Colab cell output
# demo.launch(share=True)  # use instead if you want a temporary PUBLIC link to show someone


## Optional: Fine-tuning (beyond the flowchart)

Not part of the 5-step flowchart, but a common next step once the frozen-base model plateaus: unfreeze the
top layers of MobileNetV2 and continue training with a low learning rate for a few more epochs. Only run
this after step 4/5 above, and re-run evaluation afterward if you use it.


In [ ]:
# base_model.trainable = True
#
# fine_tune_at = 100  # keep the first 100 layers frozen, unfreeze the rest
# for layer in base_model.layers[:fine_tune_at]:
#     layer.trainable = False
#
# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
#     loss='sparse_categorical_crossentropy',
#     metrics=['accuracy'],
# )
#
# history_fine = model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=10,
#     callbacks=callbacks,
# )
